# Lab 01: PyTorch Tensors, Mathematical Operations & Automatic Differentiation

Welcome to the foundational laboratory for Deep Learning! In this lab, we develop deep practical mastery of the core computational engine behind modern AI: the **PyTorch Tensor** and **Automatic Differentiation (`torch.autograd`)**.

### Learning Objectives
1. **Tensor Foundations**: Understand memory layout, data types, device placement (CPU vs. GPU), and dimensionality.
2. **Tensor Arithmetic & Broadcasting**: Master element-wise operations, broadcasting rules, matrix multiplication (`@`), and contractions.
3. **Computational Graphs & Autograd**: Understand forward graphs, backward gradient propagation, and analytical verification of partial derivatives.
4. **Multi-Variable Chain Rule**: Trace loss gradients with respect to weights and biases across composite computational graphs.


## 1. Technical Preliminaries & Hardware Setup

Before performing any tensor operations, we establish our software environment, configure random seeds to guarantee scientific reproducibility, and inspect hardware acceleration availability (NVIDIA CUDA GPU).


In [ ]:
# Import fundamental scientific computing and deep learning packages
import torch
import numpy as np
import matplotlib.pyplot as plt

# Set deterministic random seeds for PyTorch and NumPy to ensure reproducibility across runs
torch.manual_seed(42)
np.random.seed(42)

# Display environment configurations and PyTorch version
print('PyTorch Version:', torch.__version__)

# Check if an NVIDIA CUDA-compatible GPU is detected and available for acceleration
cuda_available = torch.cuda.is_available()
print('CUDA Available?', cuda_available)

# Dynamically assign execution device (GPU if available, otherwise fallback to CPU)
device = torch.device('cuda' if cuda_available else 'cpu')
print('Active Compute Device:', device)


## 2. Tensor Creation, Shapes, and Memory Allocation

### Architecture & Concept Overview: The PyTorch Tensor
A **Tensor** is a multi-dimensional array of homogeneous numerical elements. In PyTorch, tensors are optimized for high-throughput parallel computing on GPUs and automatically track computational history for backpropagation.

* **Scalar (Rank 0)**: $x \in \mathbb{R}$, zero dimensions. Shape: `torch.Size([])`
* **Vector (Rank 1)**: $\mathbf{x} \in \mathbb{R}^n$, 1 dimension. Shape: `torch.Size([n])`
* **Matrix (Rank 2)**: $\mathbf{X} \in \mathbb{R}^{m \times n}$, 2 dimensions. Shape: `torch.Size([m, n])`
* **N-D Tensor (Rank $k$)**: $\mathcal{X} \in \mathbb{R}^{d_1 \times d_2 \times \dots \times d_k}$ (e.g. Batched RGB images: `[Batch, Channels, Height, Width]`).


In [ ]:
# 1. Create a 0-dimensional scalar tensor (single numerical value)
scalar = torch.tensor(3.14159)
print('Scalar:', scalar, '| Dimensions:', scalar.ndim, '| Shape:', scalar.shape)

# 2. Create a 1-dimensional vector (array of floating-point numbers)
vector = torch.tensor([1.0, 2.0, 3.0, 4.0])
print('Vector:', vector, '| Dimensions:', vector.ndim, '| Shape:', vector.shape)

# 3. Create a 2-dimensional matrix (3 rows x 2 columns)
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
print('Matrix:\n', matrix, '\nDimensions:', matrix.ndim, '| Shape:', matrix.shape)

# 4. Factory functions for structured tensor initialization
zeros_tensor = torch.zeros((2, 3))       # All elements initialized to 0.0
ones_tensor = torch.ones((2, 3))         # All elements initialized to 1.0
rand_gaussian = torch.randn((2, 3))      # Elements sampled from Standard Normal Distribution N(0, 1)

print('\nZeros Tensor (2x3):\n', zeros_tensor)
print('Ones Tensor (2x3):\n', ones_tensor)
print('Random Gaussian Tensor (2x3):\n', rand_gaussian)


## 3. Reshaping, Slicing & Broadcasting

### Conceptual Overview: Memory Views & Broadcasting Semantics
* **`view()` vs `reshape()`**: `view()` returns a new tensor with the same underlying data without copying memory if the storage is contiguous.
* **Broadcasting Rules**: When performing element-wise arithmetic on two tensors of different shapes, PyTorch automatically aligns dimensions from right to left:
  1. If dimensions have unequal size, the dimension with size 1 is stretched/replicated to match the other.
  2. If dimensions are missing on the left, new singleton dimensions of size 1 are prepended.


In [ ]:
# Create a 1D sequence tensor of 12 integers from 0 to 11
x = torch.arange(12)
print('Original 1D tensor:', x, '| Shape:', x.shape)

# Reshape 1D tensor into a 2D matrix of shape (3 rows, 4 columns) without memory re-allocation
x_2d = x.view(3, 4)
print('\nReshaped 2D Matrix (3x4):\n', x_2d, '| Shape:', x_2d.shape)

# Add a batch dimension at index 0 using unsqueeze to convert shape from (3, 4) -> (1, 3, 4)
x_batched = x_2d.unsqueeze(0)
print('\nBatched Tensor Shape (1, 3, 4):', x_batched.shape)

# Slicing: Extract sub-matrix (first 2 rows, columns 1 to 3)
sliced_submatrix = x_2d[0:2, 1:3]
print('\nSliced Sub-matrix [0:2, 1:3]:\n', sliced_submatrix)

# Demonstrating Broadcasting: (3, 1) column vector + (1, 4) row vector -> (3, 4) result matrix
col_vec = torch.tensor([[10.0], [20.0], [30.0]]) # Shape: (3, 1)
row_vec = torch.tensor([[1.0, 2.0, 3.0, 4.0]])   # Shape: (1, 4)

# PyTorch broadcasts col_vec across columns and row_vec across rows
broadcasted_sum = col_vec + row_vec
print('\nBroadcasted Sum (3, 1) + (1, 4) -> Result Shape (3, 4):\n', broadcasted_sum)


## 4. Matrix Multiplication vs. Element-wise Operations

### Mathematical Formulation
Given matrices $\mathbf{A} \in \mathbb{R}^{M \times K}$ and $\mathbf{B} \in \mathbb{R}^{K \times N}$:
* **Element-wise Multiplication (`*`)**: Requires identical shapes (or broadcastable shapes). $C_{ij} = A_{ij} \cdot B_{ij}$.
* **Matrix Multiplication (`@` or `torch.matmul`)**: Computes the dot product of rows of $\mathbf{A}$ with columns of $\mathbf{B}$:
  $$C_{ij} = \sum_{k=1}^{K} A_{ik} B_{kj}$$
  Resulting shape is $(M \times N)$.


In [ ]:
# Define two 2x2 square matrices
A = torch.tensor([[1.0, 2.0], 
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], 
                  [7.0, 8.0]])

# 1. Element-wise (Hadamard) product: A * B -> [1*5, 2*6; 3*7, 4*8]
elem_wise = A * B
print('Element-wise Multiplication (A * B):\n', elem_wise)

# 2. True Matrix Multiplication: A @ B -> [1*5+2*7, 1*6+2*8; 3*5+4*7, 3*6+4*8] = [19, 22; 43, 50]
mat_mul = A @ B
print('\nMatrix Multiplication (A @ B):\n', mat_mul)

# Analytical verification using PyTorch unit assertion
expected = torch.tensor([[19.0, 22.0], [43.0, 50.0]])
assert torch.allclose(mat_mul, expected), 'Matrix multiplication mismatch!'
print('\n[Verification Passed] Mathematical matrix multiplication matches analytical ground truth!')


## 5. Automatic Differentiation with `torch.autograd`

### Conceptual & Mathematical Overview: The Computational Graph
PyTorch constructs a **Dynamic Directed Acyclic Graph (DAG)** of operations during the forward pass.
* Setting `requires_grad=True` instructs PyTorch to track all operations performed on this tensor.
* When `.backward()` is invoked on a scalar output $y$, PyTorch traverses the graph in reverse, applying the **Multivariate Chain Rule** to populate the `.grad` attribute of all leaf tensors:
  $$\frac{\partial y}{\partial x} = \sum_{i} \frac{\partial y}{\partial u_i} \frac{\partial u_i}{\partial x}$$

For the quadratic function $y = 3x^2 + 2x + 1$:
$$\frac{dy}{dx} = 6x + 2$$
At $x = 4.0$, $\frac{dy}{dx} = 6(4.0) + 2 = 26.0$.


In [ ]:
# Initialize leaf tensor with gradient tracking enabled
x = torch.tensor(4.0, requires_grad=True)

# Define forward computational graph: y = 3x^2 + 2x + 1
y = 3 * (x ** 2) + 2 * x + 1
print('Forward Pass: At x =', x.item(), '-> Computed y =', y.item())

# Execute backward pass (Automatic Differentiation via reverse-mode accumulation)
y.backward()

# Inspect accumulated gradient: dy/dx stored inside x.grad
computed_grad = x.grad.item()
analytical_grad = 6 * 4.0 + 2.0

print('Computed dy/dx via Autograd:', computed_grad)
print('Analytical dy/dx (6x + 2):  ', analytical_grad)

# Assert numerical equivalence between autograd and analytical calculus
assert computed_grad == analytical_grad, 'Autograd computation mismatch!'
print('\n[Verification Passed] PyTorch Autograd matches exact analytical derivative!')


## 6. Multi-Variable Gradients & Logistic Loss Backpropagation

### Mathematical Architecture: Computational Graph for a Single Neuron
Let us compute gradients for an artificial neuron with sigmoid activation and squared error loss:
1. **Linear Combination**: $z = w \cdot x + b$
2. **Sigmoid Activation**: $a = \sigma(z) = \frac{1}{1 + e^{-z}}$
3. **Squared Error Loss**: $\mathcal{L} = (a - y_{true})^2$

Using the chain rule:
$$\frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w} = 2(a - y_{true}) \cdot [a(1-a)] \cdot x$$
$$\frac{\partial \mathcal{L}}{\partial b} = \frac{\partial \mathcal{L}}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial b} = 2(a - y_{true}) \cdot [a(1-a)] \cdot 1$$


In [ ]:
# Define trainable parameters (weights and bias) and input data with gradient tracking
w = torch.tensor(2.0, requires_grad=True)   # Weight parameter
x = torch.tensor(-1.5, requires_grad=True)  # Input feature
b = torch.tensor(0.5, requires_grad=True)   # Bias parameter
target_y = 1.0                              # Target ground truth label

# Step 1: Compute affine linear transformation z = w * x + b
z = w * x + b

# Step 2: Apply non-linear Sigmoid activation function a = 1 / (1 + exp(-z))
a = torch.sigmoid(z)

# Step 3: Compute Mean Squared Error loss L = (a - y)^2
loss = (a - target_y) ** 2

print(f'Forward Values -> z: {z.item():.4f}, Activation a: {a.item():.4f}, Loss L: {loss.item():.4f}')

# Step 4: Propagate gradients backward through the entire computational graph
loss.backward()

# Display computed analytical partial derivatives
print('\n--- Autograd Computed Partial Derivatives ---')
print('dL/dw (Gradient w.r.t weight):', w.grad.item())
print('dL/dx (Gradient w.r.t input): ', x.grad.item())
print('dL/db (Gradient w.r.t bias):  ', b.grad.item())


## 7. Summary & Core Takeaways
1. **PyTorch Tensors** are GPU-accelerated n-dimensional arrays that maintain continuous memory buffers.
2. **Broadcasting** simplifies dimension alignment without explicit data duplication.
3. **`torch.autograd`** automatically computes exact derivatives through reverse-mode automatic differentiation on dynamic execution graphs.
4. **Gradient Accumulation**: PyTorch accumulates gradients in `.grad` by default; always call `optimizer.zero_grad()` or `tensor.grad.zero_()` before subsequent backward passes in training loops.
